In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/otto-recommender-system/sample_submission.csv
/kaggle/input/competitions/otto-recommender-system/test.jsonl
/kaggle/input/competitions/otto-recommender-system/train.jsonl


In [7]:
import json, itertools

DATA_DIR = "/kaggle/input/competitions/otto-recommender-system"
path = f"{DATA_DIR}/train.jsonl"

with open(path) as f:
    for line in itertools.islice(f, 3):
        s = json.loads(line)
        print(s["session"], len(s["events"]), s["events"][:3])

0 276 [{'aid': 1517085, 'ts': 1659304800025, 'type': 'clicks'}, {'aid': 1563459, 'ts': 1659304904511, 'type': 'clicks'}, {'aid': 1309446, 'ts': 1659367439426, 'type': 'clicks'}]
1 32 [{'aid': 424964, 'ts': 1659304800025, 'type': 'carts'}, {'aid': 1492293, 'ts': 1659304852871, 'type': 'clicks'}, {'aid': 1492293, 'ts': 1659304863627, 'type': 'carts'}]
2 33 [{'aid': 763743, 'ts': 1659304800038, 'type': 'clicks'}, {'aid': 137492, 'ts': 1659304866606, 'type': 'clicks'}, {'aid': 504789, 'ts': 1659304915207, 'type': 'clicks'}]


In [8]:
import pandas as pd

N = 100_000
sessions, aids, tss, types = [], [], [], []

with open(path) as f:
    for line in itertools.islice(f, N):
        s = json.loads(line)
        sid = s["session"]
        for e in s["events"]:
            sessions.append(sid)
            aids.append(e["aid"])
            tss.append(e["ts"])
            types.append(e["type"])

df = pd.DataFrame({"session": sessions, "aid": aids, "ts": tss, "type": types})
df["type"] = df["type"].astype("category")   # économise de la mémoire
print(df.shape)
df.head()

(5227653, 4)


,session,aid,ts,type
0,0,1517085,1659304800025,clicks
1,0,1563459,1659304904511,clicks
2,0,1309446,1659367439426,clicks
3,0,16246,1659367719997,clicks
4,0,1781822,1659367871344,clicks


In [9]:
print("Sessions distinctes :", df["session"].nunique())
print("Produits distincts (aid) :", df["aid"].nunique())

Sessions distinctes : 100000
Produits distincts (aid) : 663079


In [10]:
print(df["type"].value_counts())

type
clicks    4770172
carts      364579
orders      92902
Name: count, dtype: int64


In [11]:
print(df.groupby("session").size().describe())

count    100000.00000
mean         52.27653
std          76.86172
min           2.00000
25%           6.00000
50%          19.00000
75%          63.00000
max         495.00000
dtype: float64


In [12]:
clicks_df = df[df["type"] == "clicks"]
print(clicks_df["aid"].value_counts().head(10))

aid
29735      3266
832192     2898
108125     2800
1733943    2638
1603001    2588
554660     2416
166037     2132
1460571    2042
247240     2002
80222      1965
Name: count, dtype: int64


In [13]:
import numpy as np

# 1. On fixe la graine pour que les résultats soient reproductibles
rng = np.random.default_rng(42)

# 2. On mélange toutes les sessions
session_ids = df["session"].unique()
rng.shuffle(session_ids)

# 3. On sépare : 80 000 sessions pour apprendre, 20 000 pour valider
valid_ids = set(session_ids[:20_000])
fit_df   = df[~df["session"].isin(valid_ids)]   # Le train
valid_df = df[df["session"].isin(valid_ids)]    # La validation

# 4. On coupe chaque session de validation en deux
inputs, truth = {}, {}
for sid, g in valid_df.groupby("session"):
    cut = rng.integers(1, len(g))            # On choisit un point de coupure
    past, future = g.iloc[:cut], g.iloc[cut:]
    
    inputs[sid] = past                       # Le passé (ce que le modèle voit)
    truth[sid] = {                           # Le futur (ce que le modèle doit deviner)
        "clicks": future.loc[future["type"] == "clicks", "aid"].head(1).tolist(),
        "carts":  future.loc[future["type"] == "carts",  "aid"].unique().tolist(),
        "orders": future.loc[future["type"] == "orders", "aid"].unique().tolist(),
    }

# 5. On vérifie que tout s'est bien passé
print(f"Sessions d'entraînement : {len(fit_df['session'].unique())}")
print(f"Sessions de validation  : {len(inputs)}")

nb_carts  = sum(1 for t in truth.values() if len(t["carts"]) > 0)
nb_orders = sum(1 for t in truth.values() if len(t["orders"]) > 0)
print(f"Sessions avec au moins un panier caché : {nb_carts}")
print(f"Sessions avec au moins une commande cachée : {nb_orders}")

Sessions d'entraînement : 80000
Sessions de validation  : 20000
Sessions avec au moins un panier caché : 7101
Sessions avec au moins une commande cachée : 3435


In [14]:
# 1. On calcule les produits les plus populaires dans les données d'entraînement
# On sépare par type d'action
top_clicks = fit_df[fit_df["type"] == "clicks"]["aid"].value_counts().head(20).index.tolist()
top_carts  = fit_df[fit_df["type"] == "carts"]["aid"].value_counts().head(20).index.tolist()
top_orders = fit_df[fit_df["type"] == "orders"]["aid"].value_counts().head(20).index.tolist()

# 2. On crée une fonction pour calculer le Recall@20
def recall_at_20(truth, predictions):
    score_total = 0
    # On calcule le score pour chaque type d'action
    for action in ["clicks", "carts", "orders"]:
        hits = 0
        total = 0
        for sid in truth:
            # Les vrais produits que l'utilisateur a vus
            vrais_produits = set(truth[sid][action])
            if len(vrais_produits) == 0:
                continue
            
            # Les 20 produits qu'on a prédits
            preds = set(predictions[sid][action])
            
            # Combien on en a trouvé ?
            hits += len(vrais_produits.intersection(preds))
            # Le dénominateur est min(20, nombre de vrais produits)
            total += min(20, len(vrais_produits))
        
        # On calcule le score pour ce type d'action
        if total > 0:
            score_total += (hits / total) * {"clicks": 0.1, "carts": 0.3, "orders": 0.6}[action]
            
    return score_total

# 3. On fait nos prédictions (les mêmes 20 produits pour tout le monde)
predictions_baseline = {}
for sid in inputs:
    predictions_baseline[sid] = {
        "clicks": top_clicks,
        "carts":  top_carts,
        "orders": top_orders,
    }

# 4. On calcule le score !
score = recall_at_20(truth, predictions_baseline)
print(f"Score de la Baseline (Popularité) : {score:.4f}")

Score de la Baseline (Popularité) : 0.0103


In [15]:
from collections import defaultdict

# 1. On construit la matrice de co-visitation
# On utilise un dictionnaire de dictionnaires : co_visite[A][B] = nombre de fois que B suit A
co_visite = defaultdict(lambda: defaultdict(int))

# On parcourt les sessions d'entraînement
for sid, g in fit_df.groupby("session"):
    # On trie les événements par temps
    g = g.sort_values("ts")
    aids = g["aid"].values
    
    # Pour chaque paire de produits consécutifs, on incrémente le compteur
    for i in range(len(aids) - 1):
        co_visite[aids[i]][aids[i+1]] += 1

# 2. On crée une fonction pour prédire les 20 produits pour une session
def predict_covisitation(session_past):
    # On prend le dernier produit vu dans la session
    dernier_produit = session_past.iloc[-1]["aid"]
    
    # On regarde dans la matrice de co-visitation
    if dernier_produit in co_visite:
        # On trie les produits associés par nombre de co-visitations décroissant
        candidats = sorted(co_visite[dernier_produit].items(), key=lambda x: x[1], reverse=True)
        # On prend les 20 premiers
        return [aid for aid, count in candidats[:20]]
    else:
        # Si on n'a jamais vu ce produit, on retourne une liste vide
        return []

# 3. On fait nos prédictions sur la validation
predictions_covisitation = {}
for sid, past in inputs.items():
    preds = predict_covisitation(past)
    
    # Si on n'a pas 20 produits, on complète avec les populaires
    # (pour éviter les listes vides)
    preds_clicks = preds + top_clicks
    preds_carts  = preds + top_carts
    preds_orders = preds + top_orders
    
    # On garde les 20 premiers uniques
    predictions_covisitation[sid] = {
        "clicks": list(dict.fromkeys(preds_clicks))[:20],
        "carts":  list(dict.fromkeys(preds_carts))[:20],
        "orders": list(dict.fromkeys(preds_orders))[:20],
    }

# 4. On calcule le score !
score_cov = recall_at_20(truth, predictions_covisitation)
print(f"Score de la Co-visitation simple : {score_cov:.4f}")

Score de la Co-visitation simple : 0.0947


In [16]:
from collections import defaultdict

# 1. On construit une SEULE matrice globale (tous types confondus)
co_visite_global = defaultdict(lambda: defaultdict(int))

for sid, g in fit_df.groupby("session"):
    g = g.sort_values("ts")
    aids = g["aid"].values
    for i in range(len(aids) - 1):
        co_visite_global[aids[i]][aids[i+1]] += 1

# 2. Fonction de prédiction avec pondération (le dernier compte plus)
def predict_pondere(session_past, matrice, top_populaires):
    derniers_produits = session_past["aid"].values[-3:]
    
    # On définit les poids : le dernier (index -1) a un poids de 3, etc.
    # Si on a moins de 3 produits, on s'adapte.
    if len(derniers_produits) == 3:
        poids = [1, 2, 3]  # 1 pour le 3ème, 2 pour le 2ème, 3 pour le dernier
    elif len(derniers_produits) == 2:
        poids = [2, 3]
    else:
        poids = [3]
        
    scores = defaultdict(int)
    for prod, poids_prod in zip(derniers_produits, poids):
        if prod in matrice:
            for candidat, count in matrice[prod].items():
                # On multiplie le nombre de co-visitations par le poids du produit
                scores[candidat] += count * poids_prod
                
    candidats_tries = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    preds = [aid for aid, score in candidats_tries[:20]]
    
    if len(preds) < 20:
        preds = list(dict.fromkeys(preds + top_populaires))[:20]
        
    return preds

# 3. On fait les prédictions
predictions_ponderees = {}
for sid, past in inputs.items():
    predictions_ponderees[sid] = {
        "clicks": predict_pondere(past, co_visite_global, top_clicks),
        "carts":  predict_pondere(past, co_visite_global, top_carts),
        "orders": predict_pondere(past, co_visite_global, top_orders),
    }

# 4. On calcule le score
score_pondere = recall_at_20(truth, predictions_ponderees)
print(f"Score de la Co-visitation pondérée (Top 3, poids 1-2-3) : {score_pondere:.4f}")

Score de la Co-visitation pondérée (Top 3, poids 1-2-3) : 0.1218


In [17]:
# Calcul du Hit Rate : parmi les 100 candidats, le vrai produit est-il présent ?
hit_count = 0
total_count = 0

for sid in inputs:
    past = inputs[sid]
    vrais_produits = set(truth[sid]["clicks"]) # On teste sur les clics pour commencer
    
    if len(vrais_produits) == 0:
        continue
        
    # On génère 100 candidats avec notre co-visitation pondérée
    candidats = predict_pondere(past, co_visite_global, top_clicks)
    
    # On vérifie si au moins un vrai produit est dans les 100 candidats
    if len(vrais_produits.intersection(set(candidats))) > 0:
        hit_count += 1
    total_count += 1

print(f"Hit Rate @ 100 pour les clics : {hit_count / total_count:.2%}")

Hit Rate @ 100 pour les clics : 26.91%


In [18]:
def generer_candidats_avances(session_past, top_populaires, co_visite, taille=100):
    candidats = []
    
    # 1. Revisit : les produits déjà vus dans la session (très fort)
    produits_vus = session_past["aid"].unique().tolist()
    # On les met dans l'ordre inverse (les plus récents d'abord)
    candidats.extend(reversed(produits_vus))
    
    # 2. Co-visitation pondérée (Top 3)
    derniers_produits = session_past["aid"].values[-3:]
    if len(derniers_produits) == 3:
        poids = [1, 2, 3]
    elif len(derniers_produits) == 2:
        poids = [2, 3]
    else:
        poids = [3]
        
    scores_cov = defaultdict(int)
    for prod, p in zip(derniers_produits, poids):
        if prod in co_visite:
            for cand, count in co_visite[prod].items():
                scores_cov[cand] += count * p
                
    candidats_cov = [aid for aid, score in sorted(scores_cov.items(), key=lambda x: x[1], reverse=True)]
    candidats.extend(candidats_cov)
    
    # 3. Popularité pour compléter
    candidats.extend(top_populaires)
    
    # 4. On déduplique et on garde les 'taille' premiers
    return list(dict.fromkeys(candidats))[:taille]

# --- Testons le nouveau Hit Rate ---
hit_count = 0
total_count = 0

for sid in inputs:
    past = inputs[sid]
    vrais_produits = set(truth[sid]["clicks"])
    
    if len(vrais_produits) == 0:
        continue
        
    # On génère 100 candidats avec la nouvelle méthode
    candidats = generer_candidats_avances(past, top_clicks, co_visite_global, taille=100)
    
    if len(vrais_produits.intersection(set(candidats))) > 0:
        hit_count += 1
    total_count += 1

print(f"Nouveau Hit Rate @ 100 pour les clics : {hit_count / total_count:.2%}")

Nouveau Hit Rate @ 100 pour les clics : 43.62%


In [19]:
from collections import defaultdict

def generer_candidats_v2(session_past, top_populaires, co_visite, taille=100):
    candidats = []
    
    # 1. Revisit : les produits déjà vus dans la session
    candidats.extend(dict.fromkeys(session_past["aid"].values[::-1].tolist()))
    
    # 2. Co-visitation pondérée (Top 3)
    derniers_produits = session_past["aid"].values[-3:]
    if len(derniers_produits) == 3:
        poids = [1, 2, 3]
    elif len(derniers_produits) == 2:
        poids = [2, 3]
    else:
        poids = [3]
        
    scores_cov = defaultdict(int)
    for prod, p in zip(derniers_produits, poids):
        if prod in co_visite:
            for cand, count in co_visite[prod].items():
                scores_cov[cand] += count * p
                
    candidats_cov = [aid for aid, score in sorted(scores_cov.items(), key=lambda x: x[1], reverse=True)]
    candidats.extend(candidats_cov)
    
    # 3. Popularité pour compléter
    candidats.extend(top_populaires)
    
    # 4. On déduplique et on garde les 'taille' premiers
    return list(dict.fromkeys(candidats))[:taille]

In [20]:
pred_v2 = {}
for sid, past in inputs.items():
    c = generer_candidats_v2(past, top_clicks, co_visite_global)[:20]
    pred_v2[sid] = {"clicks": c, "carts": c, "orders": c}
print(f"Candidats seuls v2 : {recall_at_20(truth, pred_v2):.4f}")

Candidats seuls v2 : 0.2460


In [21]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from collections import defaultdict

rng = np.random.default_rng(42)
ACTIONS = {"clicks": 0, "carts": 1, "orders": 2}

# Popularité (recalculée ici au cas où la session aurait été relancée)
pop_counts = fit_df["aid"].value_counts().to_dict()

# 1. Séparation : sessions du ranker / sessions de la matrice
sample_sessions = list(fit_df["session"].unique())[:30000]
ranker_set = set(sample_sessions)

ranker_df = fit_df[fit_df["session"].isin(ranker_set)]
base_df   = fit_df[~fit_df["session"].isin(ranker_set)]

# 2. Matrice de co-visitation SANS les sessions du ranker (évite la fuite)
co_visite_train = defaultdict(lambda: defaultdict(int))
for sid, g in base_df.groupby("session"):
    aids = g.sort_values("ts")["aid"].values
    for i in range(len(aids) - 1):
        co_visite_train[aids[i]][aids[i + 1]] += 1

# 3. Features
def make_features(past, candidats, action_id):
    vues   = past["aid"].value_counts().to_dict()
    carts  = set(past.loc[past["type"] == "carts",  "aid"])
    orders = set(past.loc[past["type"] == "orders", "aid"])
    return pd.DataFrame({
        "rang": range(len(candidats)),
        "popularite": [pop_counts.get(a, 0) for a in candidats],
        "nb_vues": [vues.get(a, 0) for a in candidats],
        "dans_panier": [int(a in carts) for a in candidats],
        "deja_commande": [int(a in orders) for a in candidats],
        "action": action_id,
    })

# 4. Données d'entraînement (coupure aléatoire, comme à la validation)
frames = []
for sid, g in ranker_df.groupby("session"):
    g = g.sort_values("ts")
    if len(g) < 2:
        continue
    cut = rng.integers(1, len(g))
    past, future = g.iloc[:cut], g.iloc[cut:]
    candidats = generer_candidats_v2(past, top_clicks, co_visite_train)

    for action, aid_action in ACTIONS.items():
        if action == "clicks":
            # même définition que la validation : le premier clic seulement
            vrais = set(future.loc[future["type"] == "clicks", "aid"].head(1))
        else:
            vrais = set(future.loc[future["type"] == action, "aid"])
        if not vrais:
            continue
        X = make_features(past, candidats, aid_action)
        X["target"] = [int(a in vrais) for a in candidats]
        X["session"] = sid
        frames.append(X)

df_train = pd.concat(frames, ignore_index=True)
print("Lignes d'entraînement :", len(df_train))

# 5. Entraînement
features = ["rang", "popularite", "nb_vues", "dans_panier", "deja_commande", "action"]
group_sizes = df_train.groupby(["session", "action"], sort=False).size().values

model = lgb.LGBMRanker(objective="lambdarank", n_estimators=100,
                       learning_rate=0.1, random_state=42)
model.fit(df_train[features], df_train["target"], group=group_sizes)

# 6. Prédiction sur la validation (matrice complète, comme avant)
predictions_rerank = {}
for sid, past in inputs.items():
    candidats = generer_candidats_v2(past, top_clicks, co_visite_global)
    preds = {}
    for action, aid_action in ACTIONS.items():
        scores = model.predict(make_features(past, candidats, aid_action)[features])
        preds[action] = [candidats[i] for i in scores.argsort()[::-1][:20]]
    predictions_rerank[sid] = preds

print(f"Re-ranking v4 : {recall_at_20(truth, predictions_rerank):.4f}")
print("Référence à battre (candidats seuls v2) : 0.2460")

Lignes d'entraînement : 3157534
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.146890 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 394
[LightGBM] [Info] Number of data points in the train set: 3157534, number of used features: 6
Re-ranking v4 : 0.2204
Référence à battre (candidats seuls v2) : 0.2460


In [22]:
predA, predB = {}, {}
for sid, past in inputs.items():
    base = generer_candidats_v2(past, top_clicks, co_visite_global)
    carted = list(dict.fromkeys(
        past.loc[past["type"] == "carts", "aid"].values[::-1].tolist()))
    std = base[:20]
    avec_panier = list(dict.fromkeys(carted + base))[:20]
    predA[sid] = {"clicks": std, "carts": std,         "orders": avec_panier}
    predB[sid] = {"clicks": std, "carts": avec_panier, "orders": avec_panier}

print(f"Variante A (panier en tête pour orders)         : {recall_at_20(truth, predA):.4f}")
print(f"Variante B (panier en tête pour orders + carts) : {recall_at_20(truth, predB):.4f}")
print("Référence (candidats seuls v2)                  : 0.2460")

Variante A (panier en tête pour orders)         : 0.2585
Variante B (panier en tête pour orders + carts) : 0.2597
Référence (candidats seuls v2)                  : 0.2460


In [23]:
import heapq, time

co_visite_full = defaultdict(lambda: defaultdict(int))
for sid, g in df.groupby("session"):
    aids = g.sort_values("ts")["aid"].values
    for i in range(len(aids) - 1):
        co_visite_full[aids[i]][aids[i + 1]] += 1
print("Matrice construite :", len(co_visite_full), "produits")

Matrice construite : 656122 produits


In [24]:
def predire_session(events, matrice, top_pop):
    aids  = [e["aid"] for e in events]
    types = [e["type"] for e in events]

    revisits = list(dict.fromkeys(aids[::-1]))
    derniers = aids[-3:]
    poids = {3: [1, 2, 3], 2: [2, 3]}.get(len(derniers), [3])

    scores = defaultdict(int)
    for prod, p in zip(derniers, poids):
        if prod in matrice:
            for cand, count in matrice[prod].items():
                scores[cand] += count * p
    cov = [a for a, _ in heapq.nlargest(40, scores.items(), key=lambda x: x[1])]

    base   = revisits + cov + top_pop
    carted = list(dict.fromkeys(a for a, t in zip(aids[::-1], types[::-1]) if t == "carts"))

    std         = list(dict.fromkeys(base))[:20]
    avec_panier = list(dict.fromkeys(carted + base))[:20]
    return std, avec_panier

def ecrire_soumission(test_path, out_path, matrice, top_pop, limite=None):
    with open(test_path) as f, open(out_path, "w") as out:
        out.write("session_type,labels\n")
        for n, line in enumerate(f):
            if limite is not None and n >= limite:
                break
            s = json.loads(line)
            std, panier = predire_session(s["events"], matrice, top_pop)
            sid = s["session"]
            out.write(f"{sid}_clicks,{' '.join(map(str, std))}\n")
            out.write(f"{sid}_carts,{' '.join(map(str, panier))}\n")
            out.write(f"{sid}_orders,{' '.join(map(str, panier))}\n")
    return n + 1

test_path = f"{DATA_DIR}/test.jsonl"

# Test de vitesse sur 20 000 sessions
t0 = time.time()
n = ecrire_soumission(test_path, "/kaggle/working/essai.csv", co_visite_full, top_clicks, limite=20_000)
dt = time.time() - t0
print(f"{n} sessions en {dt:.0f} s -> estimation pour 1 671 803 sessions : {dt * 1_671_803 / n / 60:.0f} min")

20001 sessions en 2 s -> estimation pour 1 671 803 sessions : 3 min


In [25]:
!head -4 /kaggle/working/essai.csv

session_type,labels
12899779_clicks,59625 29735 832192 108125 1603001 1733943 554660 1460571 166037 184976 247240 332654 80222 1116095 1083665 959208 1743151 714524 1498443 673407
12899779_carts,59625 29735 832192 108125 1603001 1733943 554660 1460571 166037 184976 247240 332654 80222 1116095 1083665 959208 1743151 714524 1498443 673407
12899779_orders,59625 29735 832192 108125 1603001 1733943 554660 1460571 166037 184976 247240 332654 80222 1116095 1083665 959208 1743151 714524 1498443 673407


In [26]:
with open(test_path) as f:
    s = json.loads(next(f))
print(s["session"], s["events"])
print("59625 dans la matrice :", 59625 in co_visite_full)

12899779 [{'aid': 59625, 'ts': 1661724000278, 'type': 'clicks'}]
59625 dans la matrice : False


In [27]:
sans_voisin = 0
n = 0
with open(test_path) as f:
    for line in itertools.islice(f, 50_000):
        s = json.loads(line)
        dernier = s["events"][-1]["aid"]
        n += 1
        if dernier not in co_visite_full:
            sans_voisin += 1
print(f"Sessions dont le dernier produit n'a aucun voisin : {sans_voisin / n:.1%}")

Sessions dont le dernier produit n'a aucun voisin : 13.7%


In [28]:
n = ecrire_soumission(test_path, "/kaggle/working/submission.csv", co_visite_full, top_clicks)
print(n, "sessions écrites")

1671803 sessions écrites


In [29]:
sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub.shape)
print(sub["labels"].isna().sum(), "lignes vides")
print(sub["labels"].str.split().str.len().describe())

(5015409, 2)
0 lignes vides
count    5015409.0
mean          20.0
std            0.0
min           20.0
25%           20.0
50%           20.0
75%           20.0
max           20.0
Name: labels, dtype: float64


In [ ]:
!cd /kaggle/working && zip -q submission.zip submission.csv
# !kaggle competitions submit \
#     -c otto-recommender-system \
#     -f /kaggle/working/submission.zip \
#     -m "OTTO co-visitation baseline"